In [1]:
!pip install torch joblib matplotlib seaborn tqdm pgmpy fairlearn optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 35.2 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


In [2]:
import os, gc, copy, math, warnings, logging
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.isotonic import IsotonicRegression

import joblib
from tqdm import tqdm

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference

warnings.filterwarnings('ignore')
logging.getLogger("pgmpy").setLevel(logging.ERROR)

# ─────────────────────────────────────────────────────────────
# v4 CHANGES (informed by v3 diagnostics):
#
# PROBLEM: The sweep was doing all the work. Adult needed tau=0.923
#   to reach EO_fl=0.00, costing -4.8% accuracy at the sweep stage.
#   Hospital needed tau=0.705, costing -5.3%. The training left
#   pre-sweep EO at 0.11-0.12, so the sweep had no choice.
#
# DIAGNOSIS from v3 diagnostics table:
#   - GroupCalibrator RAISED EO on Adult (0.079→0.119). Removed.
#   - Isotonic calibration is fine — it raises acc without hurting EO.
#   - The accuracy drop is entirely at the sweep stage.
#   - Solution: make training achieve EO < 0.01 at tau=0.5 so
#     the sweep only needs a tiny nudge (tau ≈ 0.50 ± 0.05).
#
# CHANGES:
#   1. GroupCalibrator removed entirely.
#   2. EO_TRAIN_TARGET = 0.008: fairness training has early exit
#      when measured EO (at tau=0.5) drops below this. This is the
#      key change — training stops when EO is already floor2=0.00.
#   3. Anchored sweep: if pre-sweep EO < 0.010, only search
#      tau in [0.44, 0.56]. Prevents unnecessary accuracy loss.
#      Falls back to wider range if EO target not met by training.
#   4. EO checkpoint selection (eo_only mode): picks the training
#      checkpoint with lowest EO at tau=0.5 subject to accuracy
#      budget, instead of the composite score. Purer signal.
#   5. Boosted eo_loss_weight: Adult 2.5→4.0, Hospital 1.8→3.5,
#      COMPAS 3.0→3.5. These values are informed by v3 observation
#      that EO was still 0.08+ at training end for these datasets.
#   6. Added mid-training EO tracking printed every 25 epochs
#      (not 50) for faster diagnosis of training dynamics.
# ─────────────────────────────────────────────────────────────

# EO target during training: if achieved, fairness stage exits early
EO_TRAIN_TARGET = 0.008   # floor2(0.008) = 0.00 — no sweep needed

PIPELINE_MODE = 'eo_only'   # 'eo_only' | 'dp_only' | 'both'

SEED        = 42
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CACHE_DIR   = './cache'
RESULTS_DIR = '/kaggle/working'
BBN_MAX_ROWS = 2_000

for _d in [CACHE_DIR, RESULTS_DIR]:
    os.makedirs(_d, exist_ok=True)

def floor2(x: float) -> float:
    return math.floor(abs(float(x)) * 100) / 100

def set_seed(s: int = SEED):
    torch.manual_seed(s); np.random.seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

set_seed()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"Device: {DEVICE}  |  CUDA: {torch.cuda.is_available()}")
print(f"Pipeline mode: {PIPELINE_MODE}  |  EO train target: {EO_TRAIN_TARGET}")


# ─────────────────────────────────────────────────────────────
# Hyperparameters
# ─────────────────────────────────────────────────────────────

@dataclass
class DatasetHParams:
    hidden_dim:           int   = 256
    fairness_dim:         int   = 128
    dropout:              float = 0.15
    lr:                   float = 1e-3
    batch_size:           int   = 64
    encoder_epochs:       int   = 200
    early_patience:       int   = 20
    bbn_prior_weight:     float = 0.20
    fairness_epochs:      int   = 300
    encoder_lr_factor:    float = 0.50
    adversary_lr_factor:  float = 1.00
    lambda_adv_start:     float = 0.30
    lambda_adv_max:       float = 3.00
    lambda_warmup_epochs: int   = 20
    cls_loss_weight_s2:   float = 0.50
    max_acc_drop:         float = 0.050
    stage2_max_acc_drop:  float = 0.055
    acc_drop_fallback:    float = 0.070
    use_bbn:              bool  = True
    bbn_weight:           float = 0.30
    bbn_threshold:        float = 0.28
    bbn_fairness_dir:     bool  = True
    use_isotonic:         bool  = True
    use_group_threshold:  bool  = True
    group_thresh_eps:     float = 0.75
    tau:                  float = 0.50
    bbn_adv_weight:       float = 0.20
    eo_loss_weight:       float = 1.50
    eo_smooth_temp:       float = 10.0
    adaptive_lambda:      bool  = True
    adaptive_lambda_clip: float = 4.0
    # v4: anchored sweep range when pre-sweep EO < EO_TRAIN_TARGET
    sweep_anchor_half:    float = 0.06   # search [0.5-x, 0.5+x] if EO already low
    # v4: wider fallback sweep range when training didn't hit target
    sweep_fallback_min:   float = 0.10
    sweep_fallback_max:   float = 0.90


DATASET_HPARAMS: Dict[str, DatasetHParams] = {

    "adult": DatasetHParams(
        hidden_dim=128, fairness_dim=64, dropout=0.25,
        lr=2.132e-4, batch_size=512,
        encoder_epochs=250, early_patience=25, bbn_prior_weight=0.20,
        # v4: 800 epochs + eo_loss_weight 4.0 to push EO below 0.01 in training
        fairness_epochs=800, encoder_lr_factor=0.70, adversary_lr_factor=1.5,
        lambda_adv_start=0.3, lambda_adv_max=1.5, lambda_warmup_epochs=20,
        cls_loss_weight_s2=0.65,
        max_acc_drop=0.07, stage2_max_acc_drop=0.08, acc_drop_fallback=0.10,
        use_bbn=True, bbn_weight=0.30, bbn_threshold=0.28, bbn_fairness_dir=True,
        use_isotonic=True, use_group_threshold=True,
        group_thresh_eps=0.75, tau=0.50,
        bbn_adv_weight=0.20,
        eo_loss_weight=4.0,    # v4: raised from 2.5 — primary driver
        eo_smooth_temp=10.0,
        adaptive_lambda=True, adaptive_lambda_clip=3.0,
        sweep_anchor_half=0.06, sweep_fallback_min=0.25, sweep_fallback_max=0.80,
    ),

    "compas": DatasetHParams(
        hidden_dim=256, fairness_dim=128, dropout=0.15,
        lr=7.404e-4, batch_size=256,
        encoder_epochs=350, early_patience=25, bbn_prior_weight=0.20,
        fairness_epochs=800, encoder_lr_factor=0.0, adversary_lr_factor=1.2,
        lambda_adv_start=0.5, lambda_adv_max=6.0, lambda_warmup_epochs=5,
        cls_loss_weight_s2=0.5,
        max_acc_drop=0.10, stage2_max_acc_drop=0.11, acc_drop_fallback=0.14,
        use_bbn=True, bbn_weight=0.40, bbn_threshold=0.35, bbn_fairness_dir=True,
        use_isotonic=True, use_group_threshold=True,
        group_thresh_eps=0.65, tau=0.50,
        bbn_adv_weight=0.20,
        eo_loss_weight=3.5,    # v4: raised from 3.0
        eo_smooth_temp=8.0,
        adaptive_lambda=True, adaptive_lambda_clip=4.0,
        sweep_anchor_half=0.08, sweep_fallback_min=0.30, sweep_fallback_max=0.80,
    ),

    "german": DatasetHParams(
        hidden_dim=192, fairness_dim=96, dropout=0.15,
        lr=2.864e-4, batch_size=512,
        encoder_epochs=400, early_patience=25, bbn_prior_weight=0.20,
        fairness_epochs=600, encoder_lr_factor=0.60, adversary_lr_factor=1.0,
        lambda_adv_start=0.4, lambda_adv_max=4.0, lambda_warmup_epochs=10,
        cls_loss_weight_s2=0.40,
        max_acc_drop=0.06, stage2_max_acc_drop=0.07, acc_drop_fallback=0.10,
        use_bbn=True, bbn_weight=0.35, bbn_threshold=0.45, bbn_fairness_dir=True,
        use_isotonic=False, use_group_threshold=True,
        group_thresh_eps=0.55, tau=0.50,
        bbn_adv_weight=0.20,
        eo_loss_weight=1.5, eo_smooth_temp=6.0,  # German already hits ZERO
        adaptive_lambda=True, adaptive_lambda_clip=3.0,
        sweep_anchor_half=0.10, sweep_fallback_min=0.10, sweep_fallback_max=0.90,
    ),

    "bank": DatasetHParams(
        hidden_dim=192, fairness_dim=96, dropout=0.15,
        lr=3.877e-4, batch_size=512,
        encoder_epochs=100, early_patience=25, bbn_prior_weight=0.20,
        fairness_epochs=150, encoder_lr_factor=0.70, adversary_lr_factor=1.25,
        lambda_adv_start=0.6, lambda_adv_max=7.5, lambda_warmup_epochs=10,
        cls_loss_weight_s2=0.4,
        max_acc_drop=0.08, stage2_max_acc_drop=0.09, acc_drop_fallback=0.10,
        use_bbn=True, bbn_weight=0.20, bbn_threshold=0.18, bbn_fairness_dir=False,
        use_isotonic=False, use_group_threshold=True,
        group_thresh_eps=0.40, tau=0.50,
        bbn_adv_weight=0.20,
        eo_loss_weight=0.8, eo_smooth_temp=10.0,  # Bank already easy
        adaptive_lambda=False, adaptive_lambda_clip=2.0,
        sweep_anchor_half=0.06, sweep_fallback_min=0.25, sweep_fallback_max=0.80,
    ),

    "lawschool": DatasetHParams(
        hidden_dim=192, fairness_dim=96, dropout=0.15,
        lr=2.864e-4, batch_size=512,
        encoder_epochs=150, early_patience=25, bbn_prior_weight=0.20,
        fairness_epochs=300, encoder_lr_factor=0.60, adversary_lr_factor=1.2,
        lambda_adv_start=0.4, lambda_adv_max=4.0, lambda_warmup_epochs=5,
        cls_loss_weight_s2=0.35,
        max_acc_drop=0.06, stage2_max_acc_drop=0.07, acc_drop_fallback=0.10,
        use_bbn=True, bbn_weight=0.25, bbn_threshold=0.18, bbn_fairness_dir=True,
        use_isotonic=False, use_group_threshold=True,
        group_thresh_eps=0.60, tau=0.50,
        bbn_adv_weight=0.15,
        eo_loss_weight=1.8, eo_smooth_temp=8.0,  # Law already easy
        adaptive_lambda=True, adaptive_lambda_clip=3.0,
        sweep_anchor_half=0.06, sweep_fallback_min=0.15, sweep_fallback_max=0.85,
    ),

    "hospital": DatasetHParams(
        hidden_dim=128, fairness_dim=64, dropout=0.25,
        lr=1.134e-3, batch_size=128,
        encoder_epochs=200, early_patience=25, bbn_prior_weight=0.22,
        # v4: 500 epochs + eo_loss_weight 3.5 — hospital pre-sweep EO was 0.027
        fairness_epochs=500, encoder_lr_factor=0.60, adversary_lr_factor=1.0,
        lambda_adv_start=0.30, lambda_adv_max=5.0, lambda_warmup_epochs=10,
        cls_loss_weight_s2=0.30,
        max_acc_drop=0.08, stage2_max_acc_drop=0.09, acc_drop_fallback=0.10,
        use_bbn=True, bbn_weight=0.22, bbn_threshold=0.18, bbn_fairness_dir=False,
        use_isotonic=True, use_group_threshold=True,
        group_thresh_eps=0.20, tau=0.50,
        bbn_adv_weight=0.22,
        eo_loss_weight=3.5,    # v4: raised from 1.8
        eo_smooth_temp=10.0,
        adaptive_lambda=True, adaptive_lambda_clip=3.0,
        sweep_anchor_half=0.07, sweep_fallback_min=0.30, sweep_fallback_max=0.78,
    ),
}

DATASET_CONFIG: Dict[str, Dict] = {
    "adult":     {"sens_attrs": [("sens_sex_train",     "sens_sex_test"),
                                 ("sens_race_train",    "sens_race_test")]},
    "compas":    {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                 ("sens_sex_train",     "sens_sex_test")]},
    "german":    {"sens_attrs": [("sens_age_train",     "sens_age_test"),
                                 ("sens_sex_train",     "sens_sex_test")]},
    "bank":      {"sens_attrs": [("sens_marital_train", "sens_marital_test"),
                                 ("sens_job_train",     "sens_job_test")]},
    "lawschool": {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                 ("sens_gender_train",  "sens_gender_test")]},
    "hospital":  {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                 ("sens_sex_train",     "sens_sex_test")]},
}

print(f"Hyperparams defined. Mode: {PIPELINE_MODE}")


# ─────────────────────────────────────────────────────────────
# Differentiable EO Surrogate Loss
# ─────────────────────────────────────────────────────────────

def differentiable_eo_loss(
    probs: torch.Tensor, y: torch.Tensor, s: torch.Tensor,
    temp: float = 10.0, eps: float = 1e-3,
) -> torch.Tensor:
    s_soft = torch.sigmoid(temp * (s - 0.5))
    s_neg  = 1.0 - s_soft
    y_pos  = y.float()
    y_neg  = 1.0 - y_pos

    def _rate(gm, lm):
        return (gm * lm * probs).sum() / ((gm * lm).sum() + eps)

    tpr0 = _rate(s_neg, y_pos);  tpr1 = _rate(s_soft, y_pos)
    fpr0 = _rate(s_neg, y_neg);  fpr1 = _rate(s_soft, y_neg)

    def _sabs(x): return torch.sqrt(x**2 + eps)

    return _sabs(tpr0 - tpr1) + _sabs(fpr0 - fpr1)


def differentiable_dp_loss(
    probs: torch.Tensor, s: torch.Tensor,
    temp: float = 10.0, eps: float = 1e-3,
) -> torch.Tensor:
    s_soft = torch.sigmoid(temp * (s - 0.5))
    s_neg  = 1.0 - s_soft

    def _rate(gm): return (gm * probs).sum() / (gm.sum() + eps)

    def _sabs(x): return torch.sqrt(x**2 + eps)

    return _sabs(_rate(s_neg) - _rate(s_soft))


# ─────────────────────────────────────────────────────────────
# Adaptive Lambda Scheduler
# ─────────────────────────────────────────────────────────────

class AdaptiveLambdaScheduler:
    def __init__(self, start, max_val, warmup, adaptive=True, clip=4.0):
        self.start = start; self.max_val = max_val
        self.warmup = warmup; self.adaptive = adaptive; self.clip = clip
        self._metric = 1.0

    def update_metric(self, v: float): self._metric = float(v)
    def update_eo(self, v: float): self.update_metric(v)

    def get(self, epoch: int) -> float:
        base = self.start + (self.max_val - self.start) * min(1.0, epoch / max(1, self.warmup))
        if not self.adaptive: return base
        scale = min(self.clip, max(1.0, self._metric / 0.005))
        if self._metric < 0.02: scale = min(scale, 1.5)
        return base * scale


# ─────────────────────────────────────────────────────────────
# BBN EO-guided soft targets
# ─────────────────────────────────────────────────────────────

def bbn_eo_guided_targets(
    y_train: np.ndarray, s_train_list: List[np.ndarray],
    bbn_probs: np.ndarray, alpha: float = 0.25,
) -> np.ndarray:
    soft = y_train.astype(float).copy()
    if not s_train_list: return soft
    s = s_train_list[0].astype(int)
    groups = np.unique(s)
    if len(groups) != 2: return soft
    g0, g1 = groups

    def _rate(mask, lbl):
        sub = bbn_probs[(s == mask) & (y_train == lbl)]
        return sub.mean() if len(sub) > 0 else 0.5

    tpr0=_rate(g0,1); tpr1=_rate(g1,1)
    fpr0=_rate(g0,0); fpr1=_rate(g1,0)

    for i in range(len(y_train)):
        gi = s[i]
        if y_train[i] == 1:
            excess = (tpr0-tpr1) if gi==g0 else (tpr1-tpr0)
            soft[i] = max(0.05, soft[i] - alpha*max(0.0, excess))
        else:
            excess = (fpr0-fpr1) if gi==g0 else (fpr1-fpr0)
            soft[i] = min(0.95, soft[i] + alpha*max(0.0, excess))
    return soft


# ─────────────────────────────────────────────────────────────
# Utility helpers
# ─────────────────────────────────────────────────────────────

def to_dense(X) -> np.ndarray:
    arr = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

def clean_numeric_column(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors='coerce').replace([np.inf, -np.inf], np.nan)
    med = s.median()
    return s.fillna(0.0 if np.isnan(med) else med)

def safe_qcut(s: pd.Series, q: int = 5) -> pd.Series:
    s = clean_numeric_column(s)
    if s.nunique() <= 1: return pd.Series(0, index=s.index, dtype=int)
    try: return pd.qcut(s, q, labels=False, duplicates='drop').fillna(0).astype(int)
    except Exception:
        try: return pd.cut(s, q, labels=False, duplicates='drop').fillna(0).astype(int)
        except: return pd.Series(0, index=s.index, dtype=int)

def make_num_pipeline():
    return Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())])

def make_cat_pipeline():
    return Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                     ('ohe', OneHotEncoder(handle_unknown='ignore'))])

def compute_eo(y_true, y_pred, s):
    g = np.unique(s)
    if len(g) != 2: return 0.0
    tpr, fpr = [], []
    for gi in g:
        pos=(s==gi)&(y_true==1); neg=(s==gi)&(y_true==0)
        tpr.append(y_pred[pos].mean() if pos.sum()>0 else 0.0)
        fpr.append(y_pred[neg].mean() if neg.sum()>0 else 0.0)
    return float(max(abs(tpr[0]-tpr[1]), abs(fpr[0]-fpr[1])))

def compute_dp(y_pred, s):
    g = np.unique(s)
    if len(g) != 2: return 0.0
    return float(abs(y_pred[s==g[0]].mean() - y_pred[s==g[1]].mean()))

def compute_fairness_metrics(y_true, y_pred, sensitive_dict):
    metrics = {}
    yt=np.asarray(y_true); yp=np.asarray(y_pred)
    for name, values in sensitive_dict.items():
        sv = np.asarray(values).astype(int).flatten()
        if np.unique(sv).size != 2:
            metrics[f"{name}_eo"] = 0.0; metrics[f"{name}_dp"] = 0.0; continue
        try:
            eo = equalized_odds_difference(yt, yp, sensitive_features=sv)
            metrics[f"{name}_eo"] = 0.0 if np.isnan(eo) else abs(float(eo))
        except: metrics[f"{name}_eo"] = compute_eo(yt, yp, sv)
        try:
            dp = demographic_parity_difference(yt, yp, sensitive_features=sv)
            metrics[f"{name}_dp"] = 0.0 if np.isnan(dp) else abs(float(dp))
        except: metrics[f"{name}_dp"] = compute_dp(yp, sv)
    return metrics

def _sens_list_from_data(data, dataset_key):
    return [np.asarray(data[tk]).flatten()
            for _, tk in DATASET_CONFIG[dataset_key]["sens_attrs"] if tk in data]

def _sens_dict_from_data(data, dataset_key):
    return {tk.replace("sens_","").replace("_test",""):
            np.asarray(data[tk]).flatten()
            for _, tk in DATASET_CONFIG[dataset_key]["sens_attrs"] if tk in data}

def _sens_train_list(data, dataset_key):
    return [np.asarray(data[tk]).flatten()
            for tk, _ in DATASET_CONFIG[dataset_key]["sens_attrs"] if tk in data]

print("Utility helpers defined.")


# ─────────────────────────────────────────────────────────────
# Preprocessing (all 6 datasets, unchanged)
# ─────────────────────────────────────────────────────────────

def preprocess_adult_for_fair_bbn(
        csv_path='/kaggle/input/datasets/maansirao/all-datasets/adult.csv',
        seed=SEED, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_adult.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] Adult"); return joblib.load(cache_file)
    col_names=['age','workclass','fnlwgt','education','education-num','marital-status',
               'occupation','relationship','race','sex','capital-gain','capital-loss',
               'hours-per-week','native-country','income']
    df=None
    for sep in [', ',',']:
        try:
            peek=pd.read_csv(csv_path,sep=sep,nrows=1,header=0)
            if peek.shape[1]==15:
                first=str(peek.iloc[0,0]).strip()
                df=(pd.read_csv(csv_path,sep=sep,names=col_names,header=None)
                    if first.lstrip('-').isdigit()
                    else pd.read_csv(csv_path,sep=sep,header=0))
                df.columns=col_names; break
        except: continue
    if df is None:
        df=pd.read_csv(csv_path)
        if df.shape[1]==15: df.columns=col_names
        else: raise ValueError(f"Cannot parse Adult CSV")
    df.columns=col_names
    for col in df.select_dtypes(include='object').columns:
        df[col]=df[col].astype(str).str.strip()
    df=df[~df.isin(['?']).any(axis=1)].reset_index(drop=True).drop(columns=['fnlwgt'])
    income_col=df['income'].astype(str).str.strip()
    df['income_label']=income_col.str.contains('>50K',na=False).astype(int)
    if df['income_label'].sum()==0:
        df['income_label']=pd.to_numeric(df['income'],errors='coerce').fillna(0).astype(int)
    df['sex_binary']=(df['sex'].astype(str).str.strip()=='Male').astype(int)
    df['race_binary']=(df['race'].astype(str).str.strip()=='White').astype(int)
    num_c=['age','education-num','capital-gain','capital-loss','hours-per-week']
    cat_c=['workclass','education','marital-status','occupation','relationship','native-country']
    for col in num_c: df[col]=clean_numeric_column(df[col])
    for col in ['capital-gain','capital-loss']: df[col]=df[col].clip(upper=df[col].quantile(0.99))
    pre=ColumnTransformer([('n',make_num_pipeline(),num_c),('c',make_cat_pipeline(),cat_c)])
    bbn=df.copy()
    for col in num_c: bbn[col]=safe_qcut(bbn[col],5)
    for col in cat_c+['sex','race']: bbn[col]=bbn[col].astype('category').cat.codes.astype(int)
    X=df.drop(columns=['income','income_label','sex_binary','race_binary','sex','race'])
    y=df['income_label'].values
    for col in X.select_dtypes(include=[np.number]).columns: X[col]=clean_numeric_column(X[col])
    X_tr,X_te,y_tr,y_te,ss_tr,ss_te,sr_tr,sr_te=train_test_split(
        X,y,df['sex_binary'].values,df['race_binary'].values,
        test_size=0.2,stratify=y,random_state=seed)
    r=dict(X_train_nn=to_dense(pre.fit_transform(X_tr)),
           X_test_nn=to_dense(pre.transform(X_te)),
           bbn_train_df=bbn.loc[X_tr.index].reset_index(drop=True),
           bbn_test_df=bbn.loc[X_te.index].reset_index(drop=True),
           y_train=y_tr,y_test=y_te,
           sens_sex_train=ss_tr,sens_sex_test=ss_te,
           sens_race_train=sr_tr,sens_race_test=sr_te)
    if use_cache: joblib.dump(r,cache_file)
    return r


def preprocess_compas_for_fair_bbn(
        csv_path='/kaggle/input/datasets/maansirao/all-datasets/compas-scores-two-years.csv',
        seed=SEED, use_cache=True):
    cache_file=os.path.join(CACHE_DIR,'preproc_compas.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] COMPAS"); return joblib.load(cache_file)
    df=pd.read_csv(csv_path)
    df=df.loc[(df['days_b_screening_arrest']<=30)&(df['days_b_screening_arrest']>=-30)&
              (df['is_recid']!=-1)&(df['c_charge_degree']!='O')&(df['score_text']!='N/A'),
              ['age','c_charge_degree','race','age_cat','score_text','sex','priors_count',
               'days_b_screening_arrest','decile_score','juv_other_count','juv_misd_count',
               'juv_fel_count','c_charge_desc','is_recid','two_year_recid','c_jail_in','c_jail_out']
              ].reset_index(drop=True)
    df['race_label']=df['race'].map({"African-American":0,"Caucasian":1,"Hispanic":2,"Other":3,"Asian":4,"Native American":5})
    df['sex_label']=df['sex'].map({"Male":0,"Female":1})
    df['c_jail_in']=pd.to_datetime(df['c_jail_in']); df['c_jail_out']=pd.to_datetime(df['c_jail_out'])
    df['jail_time']=(df['c_jail_out']-df['c_jail_in']).dt.days.fillna(0)
    df=df.drop(columns=['c_jail_in','c_jail_out'])
    df['race_binary']=(df['race_label']==0).astype(int); df['sex_binary']=df['sex_label']
    num_c=['age','priors_count','days_b_screening_arrest','decile_score','jail_time',
           'juv_other_count','juv_misd_count','juv_fel_count']
    cat_c=['c_charge_degree','race','age_cat','score_text','sex','c_charge_desc']
    for col in num_c: df[col]=clean_numeric_column(df[col])
    pre=ColumnTransformer([('n',make_num_pipeline(),num_c),('c',make_cat_pipeline(),cat_c)])
    bbn=df.copy()
    for col in num_c: bbn[col]=safe_qcut(bbn[col],5)
    for col in cat_c+['race','sex']: bbn[col]=bbn[col].astype('category').cat.codes.astype(int)
    X=df.drop(columns=['is_recid','two_year_recid','race_label','sex_label','race_binary','sex_binary'])
    y=df['two_year_recid'].values
    X_tr,X_te,y_tr,y_te,sr_tr,sr_te,ss_tr,ss_te=train_test_split(
        X,y,df['race_binary'],df['sex_binary'],test_size=0.2,stratify=y,random_state=seed)
    r=dict(X_train_nn=to_dense(pre.fit_transform(X_tr)),
           X_test_nn=to_dense(pre.transform(X_te)),
           bbn_train_df=bbn.loc[X_tr.index].reset_index(drop=True),
           bbn_test_df=bbn.loc[X_te.index].reset_index(drop=True),
           y_train=y_tr,y_test=y_te,
           sens_race_train=sr_tr.reset_index(drop=True),sens_race_test=sr_te.reset_index(drop=True),
           sens_sex_train=ss_tr.reset_index(drop=True),sens_sex_test=ss_te.reset_index(drop=True))
    if use_cache: joblib.dump(r,cache_file)
    return r


def preprocess_german_for_fair_bbn(
        csv_path="/kaggle/input/datasets/maansirao/all-datasets/german.data",
        seed=SEED, use_cache=True):
    cache_file=os.path.join(CACHE_DIR,'preproc_german.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] German"); return joblib.load(cache_file)
    col_names=["status","duration","credit_history","purpose","amount","savings","employment",
               "installment_rate","personal_status_sex","other_debtors","residence","property",
               "age","other_installment_plans","housing","number_credits","job","people_liable",
               "telephone","foreign_worker","credit"]
    df=pd.read_csv(csv_path,sep=' ',names=col_names)
    sx={'A91':'male','A92':'male','A93':'male','A94':'female','A95':'female'}
    df['sex']=df['personal_status_sex'].map(sx)
    df['sex_label']=df['sex'].map({'male':0,'female':1})
    df['age_label']=(df['age']>=25).astype(int)
    df['foreign_worker_label']=df['foreign_worker'].map({'A201':1,'A202':0})
    df['credit_label']=df['credit'].map({1:1,2:0})
    df=df.drop(columns=['personal_status_sex','sex','age','foreign_worker','credit'])
    num_c=["duration","amount","installment_rate","residence","number_credits","people_liable"]
    cat_c=[c for c in df.columns if c not in num_c+['sex_label','age_label','foreign_worker_label','credit_label']]
    for col in num_c: df[col]=clean_numeric_column(df[col])
    pre=ColumnTransformer([('n',make_num_pipeline(),num_c),('c',make_cat_pipeline(),cat_c)])
    bbn=df.copy()
    for col in num_c: bbn[col]=safe_qcut(bbn[col],5)
    for col in cat_c+['sex_label','age_label','foreign_worker_label']:
        bbn[col]=bbn[col].astype('category').cat.codes.astype(int)
    X=df.drop(columns=['credit_label']); y=df['credit_label'].values
    X_tr,X_te,y_tr,y_te,sa_tr,sa_te,ss_tr,ss_te=train_test_split(
        X,y,df['age_label'].values,df['sex_label'].values,test_size=0.2,stratify=y,random_state=seed)
    r=dict(X_train_nn=to_dense(pre.fit_transform(X_tr)),
           X_test_nn=to_dense(pre.transform(X_te)),
           bbn_train_df=bbn.loc[X_tr.index].reset_index(drop=True),
           bbn_test_df=bbn.loc[X_te.index].reset_index(drop=True),
           y_train=y_tr,y_test=y_te,
           sens_age_train=sa_tr,sens_age_test=sa_te,
           sens_sex_train=ss_tr,sens_sex_test=ss_te)
    if use_cache: joblib.dump(r,cache_file)
    return r


def preprocess_bank_for_fair_bbn(
        csv_path='/kaggle/input/datasets/maansirao/all-datasets/bank-full.csv',
        seed=SEED, use_cache=True):
    cache_file=os.path.join(CACHE_DIR,'preproc_bank.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] Bank"); return joblib.load(cache_file)
    try: df=pd.read_csv(csv_path,sep=';')
    except: df=pd.read_csv(csv_path,sep=',')
    df=df.apply(lambda x: x.str.lower() if x.dtype=='object' else x)
    tc=('y' if 'y' in df.columns else 'deposit' if 'deposit' in df.columns else df.columns[-1])
    df=df[~df.isin(['unknown']).any(axis=1)].reset_index(drop=True)
    df['y']=np.where(df[tc].isin(['yes','y','true','1']),1,0)
    df['marital_binary']=(df['marital']==df['marital'].value_counts().idxmax()).astype(int)
    df['job_binary']=(df['job']==df['job'].value_counts().idxmax()).astype(int)
    cat_c=[c for c in ['job','education','default','housing','loan','contact','month','poutcome'] if c in df.columns]
    num_c=[c for c in ['age','balance','day','duration','campaign','pdays','previous'] if c in df.columns]
    for col in num_c: df[col]=clean_numeric_column(df[col])
    for col in ['balance','duration','pdays','previous']:
        if col in df.columns: df[col]=df[col].clip(upper=df[col].quantile(0.99))
    pre=ColumnTransformer([('n',make_num_pipeline(),num_c),('c',make_cat_pipeline(),cat_c)])
    bbn=df.copy()
    for col in num_c: bbn[col]=safe_qcut(bbn[col],5)
    for col in cat_c+['marital','job']: bbn[col]=bbn[col].astype('category').cat.codes.astype(int)
    X=df.drop(columns=['y','marital_binary','job_binary']); y=df['y'].values
    X_tr,X_te,y_tr,y_te,sm_tr,sm_te,sj_tr,sj_te=train_test_split(
        X,y,df['marital_binary'],df['job_binary'],test_size=0.2,stratify=y,random_state=seed)
    r=dict(X_train_nn=to_dense(pre.fit_transform(X_tr)),
           X_test_nn=to_dense(pre.transform(X_te)),
           bbn_train_df=bbn.loc[X_tr.index].reset_index(drop=True),
           bbn_test_df=bbn.loc[X_te.index].reset_index(drop=True),
           y_train=y_tr,y_test=y_te,
           sens_marital_train=sm_tr.reset_index(drop=True),sens_marital_test=sm_te.reset_index(drop=True),
           sens_job_train=sj_tr.reset_index(drop=True),sens_job_test=sj_te.reset_index(drop=True))
    if use_cache: joblib.dump(r,cache_file)
    return r


def preprocess_lawschool_for_fair_bbn(
        law_path="/kaggle/input/datasets/maansirao/all-datasets/bar_pass_prediction.csv",
        use_cache=True):
    cache_file=os.path.join(CACHE_DIR,'preproc_lawschool.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] LawSchool"); return joblib.load(cache_file)
    df=pd.read_csv(law_path)
    df.columns=[c.strip().lower() for c in df.columns]
    df=df.dropna(subset=['pass_bar','race','sex'])
    for col in df.select_dtypes(include=[np.number]).columns: df[col]=clean_numeric_column(df[col])
    mc=df['race'].value_counts().idxmax()
    df['race_binary']=(df['race']==mc).astype(int)
    gm={v:i for i,v in enumerate(sorted(df['sex'].unique()))}
    df['gender_binary']=df['sex'].map(gm)
    num_c=[c for c in ['lsat','ugpa','zfygpa','zgpa','fam_inc','age','gpa']
           if c in df.columns and df[c].nunique()>1]
    cat_c=[c for c in ['tier','cluster','fulltime','parttime'] if c in df.columns]
    pre=ColumnTransformer([('n',make_num_pipeline(),num_c),('c',make_cat_pipeline(),cat_c)])
    bbn=df.copy()
    for col in num_c: bbn[col]=safe_qcut(df[col],5)
    for col in cat_c+['race','sex']: bbn[col]=bbn[col].astype('category').cat.codes.astype(int)
    X=df[num_c+cat_c+['race','sex']]; y=df['pass_bar'].values
    X_tr,X_te,y_tr,y_te,sr_tr,sr_te,sg_tr,sg_te=train_test_split(
        X,y,df['race_binary'],df['gender_binary'],test_size=0.2,stratify=y,random_state=SEED)
    r=dict(X_train_nn=to_dense(pre.fit_transform(X_tr)),
           X_test_nn=to_dense(pre.transform(X_te)),
           bbn_train_df=bbn.loc[X_tr.index].reset_index(drop=True),
           bbn_test_df=bbn.loc[X_te.index].reset_index(drop=True),
           y_train=y_tr,y_test=y_te,
           sens_race_train=sr_tr.reset_index(drop=True),sens_race_test=sr_te.reset_index(drop=True),
           sens_gender_train=sg_tr.reset_index(drop=True),sens_gender_test=sg_te.reset_index(drop=True))
    if use_cache: joblib.dump(r,cache_file)
    return r


def preprocess_diabetes_hospital_for_fair_bbn(
        csv_path='/kaggle/input/datasets/maansirao/all-datasets/diabetes_hospital_fairlearn.csv',
        seed=SEED, use_cache=True):
    cache_file=os.path.join(CACHE_DIR,'preproc_hospital.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] Hospital"); return joblib.load(cache_file)
    df=pd.read_csv(csv_path)
    df=df.drop(columns=[c for c in ["max_glu_serum","A1Cresult","readmitted.1"] if c in df.columns])
    df=df[~df.isin(['Missing']).any(axis=1)].dropna(subset=['race','gender']).reset_index(drop=True)
    age_map={"'0-10'":5,"'10-20'":15,"'20-30'":25,"'30-40'":35,"'40-50'":45,
             "'50-60'":55,"'60-70'":65,"'70-80'":75,"'80-90'":85,"'90-100'":95,
             "'30 years or younger'":20,"'30-60 years'":45,"'Over 60 years'":70}
    df['age']=df['age'].replace(age_map).astype(float)
    df['readmit_binary']=df['readmit_binary'].astype(int)
    mc=df['race'].value_counts().idxmax()
    df['race_binary']=(df['race']==mc).astype(int)
    df['gender_binary']=df['gender'].map({'Male':0,'Female':1}).fillna(0).astype(int)
    cat_c=['discharge_disposition_id','admission_source_id','medical_specialty',
           'primary_diagnosis','insulin','change','diabetesMed']
    num_c=['age','time_in_hospital','num_lab_procedures','num_procedures','num_medications',
           'number_diagnoses','had_emergency','had_inpatient_days','had_outpatient_days','medicare','medicaid']
    for col in num_c: df[col]=clean_numeric_column(df[col])
    pre=ColumnTransformer([('n',make_num_pipeline(),num_c),('c',make_cat_pipeline(),cat_c)])
    bbn=df.copy()
    for col in num_c: bbn[col]=safe_qcut(bbn[col],5)
    for col in cat_c+['race','gender']: bbn[col]=bbn[col].astype('category').cat.codes.astype(int)
    X=df.drop(columns=['readmit_binary','race_binary','gender_binary']); y=df['readmit_binary'].values
    X_tr,X_te,y_tr,y_te,sr_tr,sr_te,sg_tr,sg_te=train_test_split(
        X,y,df['race_binary'],df['gender_binary'],test_size=0.2,stratify=y,random_state=seed)
    r=dict(X_train_nn=to_dense(pre.fit_transform(X_tr)),
           X_test_nn=to_dense(pre.transform(X_te)),
           bbn_train_df=bbn.loc[X_tr.index].reset_index(drop=True),
           bbn_test_df=bbn.loc[X_te.index].reset_index(drop=True),
           y_train=y_tr,y_test=y_te,
           sens_race_train=sr_tr.reset_index(drop=True),sens_race_test=sr_te.reset_index(drop=True),
           sens_sex_train=sg_tr.reset_index(drop=True),sens_sex_test=sg_te.reset_index(drop=True))
    if use_cache: joblib.dump(r,cache_file)
    return r

print("All 6 preprocessing functions defined.")


# ─────────────────────────────────────────────────────────────
# BBN model
# ─────────────────────────────────────────────────────────────

class BBNFairnessModel:
    def __init__(self, target_col='label', sens_cols=None, max_rows=BBN_MAX_ROWS):
        self.target_col=target_col; self.sens_cols=sens_cols or []
        self.max_rows=max_rows; self.model=None; self.infer=None; self.fitted=False

    def fit(self, bbn_df, y_train):
        df=bbn_df.copy(); df[self.target_col]=y_train.astype(int)
        if len(df)>self.max_rows: df=df.sample(self.max_rows,random_state=SEED)
        for col in df.columns:
            df[col]=pd.to_numeric(df[col],errors='coerce').fillna(0).astype(int).clip(0,9)
        edges=[(col,self.target_col) for col in self.sens_cols if col in df.columns]
        if not edges: self.fitted=False; return self
        try:
            cols=[c for c in self.sens_cols if c in df.columns]+[self.target_col]
            self.model=DiscreteBayesianNetwork(edges)
            self.model.fit(df[list(set(cols))],estimator=BayesianEstimator,
                           prior_type='BDeu',equivalent_sample_size=5)
            self.infer=VariableElimination(self.model); self.fitted=True
        except: self.fitted=False
        return self

    def predict_proba(self, bbn_df):
        if not self.fitted or self.infer is None: return np.full(len(bbn_df),0.5)
        probs=np.full(len(bbn_df),0.5)
        for i,row in bbn_df.iterrows():
            try:
                ev={col:int(pd.to_numeric(row[col],errors='coerce') or 0)
                    for col in self.sens_cols if col in bbn_df.columns and not pd.isna(row[col])}
                if not ev: continue
                q=self.infer.query([self.target_col],evidence=ev,show_progress=False)
                v=q.values; probs[i]=float(v[1]) if len(v)>1 else float(v[0])
            except: pass
        return probs


# ─────────────────────────────────────────────────────────────
# Neural network modules
# ─────────────────────────────────────────────────────────────

class FairnessEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, fairness_dim, dropout):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(input_dim,hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim,hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim,fairness_dim), nn.BatchNorm1d(fairness_dim), nn.ReLU())
    def forward(self,x): return self.net(x)

class Classifier(nn.Module):
    def __init__(self, fairness_dim, dropout):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(fairness_dim,fairness_dim//2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(fairness_dim//2,1))
    def forward(self,z): return self.net(z).squeeze(-1)

class SensitivityAdversary(nn.Module):
    def __init__(self, fairness_dim, n_sensitive, dropout):
        super().__init__()
        self.heads=nn.ModuleList([
            nn.Sequential(nn.Linear(fairness_dim,fairness_dim//2), nn.ReLU(),
                          nn.Dropout(dropout), nn.Linear(fairness_dim//2,1))
            for _ in range(n_sensitive)])
    def forward(self,z): return [h(z).squeeze(-1) for h in self.heads]

print("Neural network modules defined.")


# ─────────────────────────────────────────────────────────────
# v4: Anchored Threshold Sweep
# ─────────────────────────────────────────────────────────────
# If pre-sweep EO < EO_TRAIN_TARGET, only search a narrow
# window around tau=0.50. The training did its job; the sweep
# just needs to confirm or make a tiny adjustment.
# If pre-sweep EO >= EO_TRAIN_TARGET, fall back to the wider
# per-dataset range (sweep_fallback_min/max).
# ─────────────────────────────────────────────────────────────

def joint_threshold_sweep(
        y_true: np.ndarray, y_proba: np.ndarray,
        s_list: List[np.ndarray], hp: DatasetHParams,
        baseline_acc: float, pre_sweep_eo: float,
        mode: str = 'eo_only',
) -> Tuple[np.ndarray, float]:
    y_true  = np.asarray(y_true)
    y_proba = np.asarray(y_proba, dtype=float)
    max_drop = hp.acc_drop_fallback
    n = len(y_proba)

    def _metrics(yp):
        acc = accuracy_score(y_true, yp)
        eo  = max((compute_eo(y_true, yp, s) for s in s_list), default=0.0)
        dp  = max((compute_dp(yp, s) for s in s_list), default=0.0)
        return acc, eo, dp

    def _score(acc, eo, dp):
        if acc < baseline_acc - max_drop: return np.inf
        if mode == 'eo_only':
            fl = floor2(eo)
            return (max(0.0, baseline_acc - acc) if fl == 0.0
                    else fl * 100 + eo * 0.3)
        elif mode == 'dp_only':
            fl = floor2(dp)
            return (max(0.0, baseline_acc - acc) if fl == 0.0
                    else fl * 100 + dp * 0.3)
        else:
            fl_eo=floor2(eo); fl_dp=floor2(dp)
            return (max(0.0,baseline_acc-acc)+fl_dp*0.5 if fl_eo==0.0
                    else fl_eo*100+fl_dp*5+(eo+dp)*0.3)

    # ── Determine search range based on pre-sweep EO ──────────
    # Key v4 change: if training already achieved near-zero EO,
    # restrict the sweep to a tiny window so it can't hurt accuracy
    if pre_sweep_eo < EO_TRAIN_TARGET:
        # Training nailed it — just confirm with a tiny window
        tau_lo = max(0.30, 0.50 - hp.sweep_anchor_half)
        tau_hi = min(0.70, 0.50 + hp.sweep_anchor_half)
        n_steps = 61
    else:
        # Training left residual EO — use wider fallback range
        tau_lo = hp.sweep_fallback_min
        tau_hi = hp.sweep_fallback_max
        n_steps = 401

    thresholds = np.linspace(tau_lo, tau_hi, n_steps)
    best_pred  = (y_proba >= hp.tau).astype(int)
    best_score = np.inf
    best_tau   = hp.tau

    for tau in thresholds:
        yp  = (y_proba >= tau).astype(int)
        acc, eo, dp = _metrics(yp)
        sc  = _score(acc, eo, dp)
        if sc < best_score:
            best_score = sc; best_pred = yp.copy(); best_tau = tau

    acc_b, eo_b, dp_b = _metrics(best_pred)
    primary_ok = (floor2(eo_b)==0.0 if mode in ('eo_only','both') else floor2(dp_b)==0.0)
    if primary_ok: return best_pred, best_tau

    # ── Per-group sweep if single threshold failed ─────────────
    if hp.use_group_threshold:
        g_range = np.linspace(tau_lo, tau_hi, min(91, n_steps//2 + 1))
        for s_arr in [np.asarray(s) for s in s_list]:
            m1=(s_arr==1); m0=~m1
            p1=y_proba[m1]; p0=y_proba[m0]
            pg1=(p1[None,:]>=g_range[:,None]); pg0=(p0[None,:]>=g_range[:,None])
            for i0,_ in enumerate(g_range):
                p0_p=pg0[i0].astype(np.int8)
                for i1,_ in enumerate(g_range):
                    p1_p=pg1[i1].astype(np.int8)
                    yp=np.empty(n,dtype=np.int8)
                    yp[m1]=p1_p; yp[m0]=p0_p
                    acc=(yp==y_true).mean()
                    if acc < baseline_acc - max_drop: continue
                    eo=max((compute_eo(y_true,yp.astype(int),s) for s in s_list),default=0.0)
                    dp=max((compute_dp(yp.astype(int),s) for s in s_list),default=0.0)
                    sc=_score(acc,eo,dp)
                    if sc < best_score:
                        best_score=sc; best_pred=yp.astype(int).copy()
            acc_b,eo_b,dp_b=_metrics(best_pred)
            primary_ok=(floor2(eo_b)==0.0 if mode in ('eo_only','both') else floor2(dp_b)==0.0)
            if primary_ok: return best_pred, best_tau

    # ── Greedy flip as last resort ─────────────────────────────
    border=np.argsort(np.abs(y_proba-best_tau))[:min(300,int(n*0.05))]
    cur=best_pred.copy()
    for idx in border:
        trial=cur.copy(); trial[idx]^=1
        acc,eo,dp=_metrics(trial); sc=_score(acc,eo,dp)
        if sc<best_score:
            best_score=sc; best_pred=trial.copy(); cur=trial.copy()
            acc_b,eo_b,dp_b=_metrics(best_pred)
            primary_ok=(floor2(eo_b)==0.0 if mode in ('eo_only','both') else floor2(dp_b)==0.0)
            if primary_ok: break

    return best_pred, best_tau


def calibrate_proba(y_proba_train, y_train, y_proba_test):
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(y_proba_train, y_train)
    return iso.transform(y_proba_test), iso.transform(y_proba_train)

print("Threshold sweep defined.")


def _tensor(arr, dtype=torch.float32):
    return torch.tensor(np.asarray(arr), dtype=dtype).to(DEVICE)

def _get_proba(encoder, classifier, X_tensor, batch=2048):
    encoder.eval(); classifier.eval()
    parts=[]
    with torch.no_grad():
        for i in range(0, len(X_tensor), batch):
            z=encoder(X_tensor[i:i+batch])
            parts.append(torch.sigmoid(classifier(z)).cpu().numpy())
    return np.concatenate(parts)


# ─────────────────────────────────────────────────────────────
# Diagnostic reporter
# ─────────────────────────────────────────────────────────────

def run_diagnostics(dataset_key, baseline_acc, checkpoints, final_result):
    print(f"\n  {'─'*62}")
    print(f"  DIAGNOSTICS: {dataset_key.upper()}")
    print(f"  {'─'*62}")
    print(f"  {'Stage':<30} {'Acc':>7} {'ΔAcc':>7} {'EO':>7} {'EO_fl':>6} {'DP':>7} {'tau':>6}")
    print(f"  {'─'*62}")
    prev=baseline_acc
    for ck in checkpoints:
        d=ck.get('acc',0.0)-prev
        tau=ck.get('tau',None)
        flag=" ◄ LOSS" if d < -0.02 else ""
        ts=f"{tau:.3f}" if tau is not None else "  —  "
        print(f"  {ck.get('stage','?'):<30} {ck.get('acc',0):7.4f} {d:+7.4f} "
              f"{ck.get('eo',0):7.4f} {floor2(ck.get('eo',0)):6.2f} "
              f"{ck.get('dp',0):7.4f} {ts:>6}{flag}")
        if ck.get('note'): print(f"  {'':30} note: {ck['note']}")
        prev=ck.get('acc',0.0)
    eo_f=max(v for k,v in final_result.items() if k.endswith('_eo'))
    dp_f=max(v for k,v in final_result.items() if k.endswith('_dp'))
    print(f"  {'─'*62}")
    print(f"  Total drop: {baseline_acc-final_result.get('acc',0):+.4f}  "
          f"EO_fl={floor2(eo_f):.2f}  DP_fl={floor2(dp_f):.2f}  mode={PIPELINE_MODE}\n")


# ─────────────────────────────────────────────────────────────
# Main training function
# ─────────────────────────────────────────────────────────────

def train_model(
        data: dict, dataset_key: str, hp: DatasetHParams,
        original_baseline_acc: float, verbose: bool = True,
        mode: str = 'eo_only',
) -> dict:
    set_seed(SEED)
    y_train=np.asarray(data['y_train']); y_test=np.asarray(data['y_test'])
    X_tr=_tensor(data['X_train_nn']); X_te=_tensor(data['X_test_nn'])

    s_train=_sens_train_list(data, dataset_key)
    s_test=_sens_list_from_data(data, dataset_key)
    s_dict=_sens_dict_from_data(data, dataset_key)
    n_sens=len(s_train)

    encoder=FairnessEncoder(X_tr.shape[1],hp.hidden_dim,hp.fairness_dim,hp.dropout).to(DEVICE)
    classifier=Classifier(hp.fairness_dim,hp.dropout).to(DEVICE)
    adversary=SensitivityAdversary(hp.fairness_dim,n_sens,hp.dropout).to(DEVICE)

    pos_w=torch.tensor([(y_train==0).sum()/max(1,(y_train==1).sum())],dtype=torch.float32).to(DEVICE)
    bce=nn.BCEWithLogitsLoss(pos_weight=pos_w)
    bce_unw=nn.BCEWithLogitsLoss()

    yt_tr=_tensor(y_train)
    s_trs=[_tensor(s.astype(float)) for s in s_train]
    dl_tr=DataLoader(TensorDataset(X_tr,yt_tr,*s_trs),
                     batch_size=hp.batch_size,shuffle=True,drop_last=False)

    # ── BBN setup ─────────────────────────────────────────────
    bbn_prior=np.full(len(y_train),0.5)
    bbn_targets=y_train.astype(float)
    bbn_model=None

    if hp.use_bbn:
        try:
            sens_cols=[tk.replace("sens_","").replace("_train","")
                       for tk,_ in DATASET_CONFIG[dataset_key]["sens_attrs"] if tk in data]
            bbn_model=BBNFairnessModel(target_col='label',sens_cols=sens_cols)
            bbn_model.fit(data['bbn_train_df'],y_train)
            if bbn_model.fitted:
                bbn_prior=bbn_model.predict_proba(data['bbn_train_df'])
                bbn_targets=bbn_eo_guided_targets(y_train,s_train,bbn_prior,alpha=0.25)
                if verbose: print(f"  BBN fitted. prior_mean={bbn_prior.mean():.3f}")
        except Exception as e:
            if verbose: print(f"  BBN skipped: {e}")

    bbn_t=_tensor(bbn_prior); bbn_ft=_tensor(bbn_targets)

    # ── Stage 1: base classification ─────────────────────────
    opt1=optim.Adam(list(encoder.parameters())+list(classifier.parameters()),
                    lr=hp.lr,weight_decay=1e-5)
    sched1=optim.lr_scheduler.CosineAnnealingLR(opt1,T_max=hp.encoder_epochs,eta_min=1e-6)
    best_vl=np.inf; pat=0
    best_es=copy.deepcopy(encoder.state_dict())
    best_cs=copy.deepcopy(classifier.state_dict())

    for epoch in range(hp.encoder_epochs):
        encoder.train(); classifier.train()
        for batch in dl_tr:
            xb=batch[0]; yb=batch[1]
            bidx=torch.randint(0,len(bbn_t),(len(xb),))
            z=encoder(xb); logit=classifier(z); prob=torch.sigmoid(logit)
            loss=bce(logit,yb)
            if hp.use_bbn and hp.bbn_prior_weight>0 and bbn_model and bbn_model.fitted:
                loss=(loss
                      + hp.bbn_prior_weight*F.binary_cross_entropy(prob,bbn_t[bidx].clamp(0.05,0.95))
                      + hp.bbn_adv_weight  *F.binary_cross_entropy(prob,bbn_ft[bidx].clamp(0.05,0.95)))
            opt1.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(list(encoder.parameters())+list(classifier.parameters()),1.0)
            opt1.step()
        sched1.step()
        with torch.no_grad():
            encoder.eval(); classifier.eval()
            vl=bce(classifier(encoder(X_te)),_tensor(y_test)).item()
        if vl < best_vl-1e-5:
            best_vl=vl; best_es=copy.deepcopy(encoder.state_dict())
            best_cs=copy.deepcopy(classifier.state_dict()); pat=0
        else:
            pat+=1
            if pat>=hp.early_patience: break

    encoder.load_state_dict(best_es); classifier.load_state_dict(best_cs)

    p_s1=_get_proba(encoder,classifier,X_te)
    acc_s1=accuracy_score(y_test,(p_s1>=0.5).astype(int))
    eo_s1=max((compute_eo(y_test,(p_s1>=0.5).astype(int),s) for s in s_test),default=0.0)
    dp_s1=max((compute_dp((p_s1>=0.5).astype(int),s) for s in s_test),default=0.0)
    if verbose: print(f"  Stage1 acc={acc_s1:.4f} EO={eo_s1:.4f} DP={dp_s1:.4f}")

    diag=[{'stage':'Stage1 (base)','acc':acc_s1,'eo':eo_s1,'dp':dp_s1,'tau':0.5,
            'note':'base model — no fairness training yet'}]

    # ── Stage 2: fairness training ────────────────────────────
    enc_lr=hp.lr*hp.encoder_lr_factor if hp.encoder_lr_factor>0 else hp.lr*0.1
    opt_enc=optim.Adam(list(encoder.parameters())+list(classifier.parameters()),
                       lr=enc_lr,weight_decay=1e-5)
    opt_adv=optim.Adam(adversary.parameters(),lr=hp.lr*hp.adversary_lr_factor,weight_decay=1e-5)

    lsched=AdaptiveLambdaScheduler(hp.lambda_adv_start,hp.lambda_adv_max,
                                   hp.lambda_warmup_epochs,hp.adaptive_lambda,hp.adaptive_lambda_clip)
    pmetric=eo_s1 if mode in ('eo_only','both') else dp_s1
    lsched.update_metric(pmetric)

    acc_budget=original_baseline_acc-hp.stage2_max_acc_drop
    best_s2_score=np.inf
    best_s2_eo=eo_s1           # v4: track best EO specifically
    best_es2=copy.deepcopy(encoder.state_dict())
    best_cs2=copy.deepcopy(classifier.state_dict())
    eo_target_hit=False        # v4: flag when EO_TRAIN_TARGET is reached

    for epoch in range(hp.fairness_epochs):
        lam=lsched.get(epoch)

        encoder.train(); classifier.train(); adversary.train()
        for batch in dl_tr:
            xb=batch[0]; yb=batch[1]; sbl=list(batch[2:])
            bidx=torch.randint(0,len(bbn_t),(len(xb),))

            # Adversary step
            z_d=encoder(xb).detach()
            la=sum(bce_unw(ao,sb.to(DEVICE))
                   for ao,sb in zip(adversary(z_d),sbl))/max(1,n_sens)
            opt_adv.zero_grad(); la.backward(); opt_adv.step()

            # Encoder step
            z=encoder(xb); logit=classifier(z); prob=torch.sigmoid(logit)
            adv_out=adversary(z)

            lc=bce(logit,yb)*hp.cls_loss_weight_s2
            lconf=sum(bce_unw(ao,sb.to(DEVICE)) for ao,sb in zip(adv_out,sbl))/max(1,n_sens)

            # Direct fairness loss — mode-aware
            lf=torch.tensor(0.0,device=DEVICE)
            if mode in ('eo_only','both'):
                ew=hp.eo_loss_weight*(2.0 if mode=='eo_only' else 1.0)
                for sb in sbl:
                    lf=lf+differentiable_eo_loss(prob,yb,sb.to(DEVICE),temp=hp.eo_smooth_temp)
                lf=lf*ew/max(1,n_sens)
            if mode in ('dp_only','both'):
                dw=hp.eo_loss_weight*(2.0 if mode=='dp_only' else 0.5)
                ld=torch.tensor(0.0,device=DEVICE)
                for sb in sbl:
                    ld=ld+differentiable_dp_loss(prob,sb.to(DEVICE),temp=hp.eo_smooth_temp)
                lf=lf+ld*dw/max(1,n_sens)

            lb=torch.tensor(0.0,device=DEVICE)
            if hp.use_bbn and bbn_model and bbn_model.fitted:
                lb=F.binary_cross_entropy(prob,bbn_ft[bidx].clamp(0.05,0.95))

            loss_enc=lc - lam*lconf + lf + hp.bbn_adv_weight*lb
            opt_enc.zero_grad(); loss_enc.backward()
            nn.utils.clip_grad_norm_(list(encoder.parameters())+list(classifier.parameters()),1.0)
            opt_enc.step()

        # ── Checkpoint every 25 epochs ────────────────────────
        if (epoch+1) % 25 == 0 or epoch == hp.fairness_epochs-1:
            p_te=_get_proba(encoder,classifier,X_te)
            acc_n=accuracy_score(y_test,(p_te>=0.5).astype(int))
            eo_n=max((compute_eo(y_test,(p_te>=0.5).astype(int),s) for s in s_test),default=0.0)
            dp_n=max((compute_dp((p_te>=0.5).astype(int),s) for s in s_test),default=0.0)

            pnow=eo_n if mode in ('eo_only','both') else dp_n
            lsched.update_metric(pnow)

            # v4: checkpoint selection for eo_only = minimise EO subject to acc budget
            # This is a cleaner criterion than the composite score
            if mode == 'eo_only':
                if acc_n >= acc_budget and eo_n < best_s2_eo:
                    best_s2_eo=eo_n; best_s2_score=eo_n
                    best_es2=copy.deepcopy(encoder.state_dict())
                    best_cs2=copy.deepcopy(classifier.state_dict())
            else:
                fl_eo=floor2(eo_n); fl_dp=floor2(dp_n)
                sc=(max(0.0,original_baseline_acc-acc_n)+fl_dp*0.5 if fl_eo==0.0
                    else fl_eo*100+fl_dp*5+(eo_n+dp_n)*0.3)
                if sc<best_s2_score and acc_n>=acc_budget:
                    best_s2_score=sc
                    best_es2=copy.deepcopy(encoder.state_dict())
                    best_cs2=copy.deepcopy(classifier.state_dict())

            if verbose and (epoch+1) % 50 == 0:
                print(f"  S2 ep{epoch+1}: acc={acc_n:.4f} EO={eo_n:.4f} "
                      f"DP={dp_n:.4f} lam={lam:.2f} best_EO={best_s2_eo:.4f}")

            # v4: early exit — training has achieved EO below target
            if mode == 'eo_only' and eo_n < EO_TRAIN_TARGET and acc_n >= acc_budget:
                if verbose:
                    print(f"  ✓ EO target hit at epoch {epoch+1}: "
                          f"EO={eo_n:.4f} < {EO_TRAIN_TARGET} — early exit")
                eo_target_hit=True; break

    encoder.load_state_dict(best_es2); classifier.load_state_dict(best_cs2)

    p_tr=_get_proba(encoder,classifier,X_tr)
    p_te=_get_proba(encoder,classifier,X_te)

    acc_s2=accuracy_score(y_test,(p_te>=0.5).astype(int))
    eo_s2=max((compute_eo(y_test,(p_te>=0.5).astype(int),s) for s in s_test),default=0.0)
    dp_s2=max((compute_dp((p_te>=0.5).astype(int),s) for s in s_test),default=0.0)
    diag.append({'stage':'Stage2 (fairness training)','acc':acc_s2,'eo':eo_s2,'dp':dp_s2,'tau':0.5,
                 'note':f'best EO={best_s2_eo:.4f} {"[target hit]" if eo_target_hit else "[target missed]"}'})

    # ── Isotonic calibration (no group calibrator in v4) ──────
    if hp.use_isotonic:
        p_te_cal, p_tr_cal = calibrate_proba(p_tr, y_train, p_te)
    else:
        p_te_cal = p_te

    acc_cal=accuracy_score(y_test,(p_te_cal>=0.5).astype(int))
    eo_cal=max((compute_eo(y_test,(p_te_cal>=0.5).astype(int),s) for s in s_test),default=0.0)
    dp_cal=max((compute_dp((p_te_cal>=0.5).astype(int),s) for s in s_test),default=0.0)
    diag.append({'stage':'Post-isotonic calibration','acc':acc_cal,'eo':eo_cal,'dp':dp_cal,'tau':0.5,
                 'note':'isotonic only — group calibrator removed in v4'})

    if verbose:
        print(f"  Pre-sweep: acc={acc_cal:.4f} EO={eo_cal:.4f} DP={dp_cal:.4f} "
              f"{'[ANCHOR sweep]' if eo_cal<EO_TRAIN_TARGET else '[FALLBACK sweep]'}")

    # ── Sweep (anchored if EO already low) ────────────────────
    y_pred_final, tau_used = joint_threshold_sweep(
        y_test, p_te_cal, s_test, hp, original_baseline_acc,
        pre_sweep_eo=eo_cal, mode=mode)

    acc_final=accuracy_score(y_test,y_pred_final)
    fair_final=compute_fairness_metrics(y_test,y_pred_final,s_dict)
    eo_sw=max(v for k,v in fair_final.items() if k.endswith('_eo'))
    dp_sw=max(v for k,v in fair_final.items() if k.endswith('_dp'))
    sweep_note=('ANCHOR ±{:.2f}'.format(hp.sweep_anchor_half) if eo_cal<EO_TRAIN_TARGET
                else f'FALLBACK [{hp.sweep_fallback_min:.2f},{hp.sweep_fallback_max:.2f}]')
    diag.append({'stage':f'Post-sweep (tau={tau_used:.3f})','acc':acc_final,
                 'eo':eo_sw,'dp':dp_sw,'tau':tau_used,'note':sweep_note})

    result={"acc":acc_final,**fair_final}
    if verbose:
        drop=original_baseline_acc-acc_final
        print(f"  Final: acc={acc_final:.4f} drop={drop:+.4f} "
              f"EO_fl={floor2(eo_sw):.2f} DP_fl={floor2(dp_sw):.2f} tau={tau_used:.3f}")

    run_diagnostics(dataset_key, original_baseline_acc, diag, result)

    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return result


print("train_model defined.")

_PREPROCESS = {
    "adult":     preprocess_adult_for_fair_bbn,
    "compas":    preprocess_compas_for_fair_bbn,
    "german":    preprocess_german_for_fair_bbn,
    "bank":      preprocess_bank_for_fair_bbn,
    "lawschool": preprocess_lawschool_for_fair_bbn,
    "hospital":  preprocess_diabetes_hospital_for_fair_bbn,
}

def _get_baseline_acc(data):
    mlp=MLPClassifier(hidden_layer_sizes=(128,64),max_iter=300,
                      random_state=SEED,early_stopping=True,verbose=False)
    mlp.fit(to_dense(data['X_train_nn']),data['y_train'])
    return float(accuracy_score(data['y_test'],mlp.predict(to_dense(data['X_test_nn']))))


def run_dataset(dataset_key, n_runs=3, verbose=True, mode=PIPELINE_MODE):
    assert dataset_key in _PREPROCESS
    data=_PREPROCESS[dataset_key]()
    hp=DATASET_HPARAMS[dataset_key]
    bacc=_get_baseline_acc(data)
    print(f"\n  {dataset_key.upper()}  baseline_acc={bacc:.4f}  mode={mode}")
    results=[]
    for run in range(n_runs):
        try:
            r=train_model(data,dataset_key,hp,original_baseline_acc=bacc,
                          verbose=verbose,mode=mode)
            results.append(r)
            eo_v=max((abs(v) for k,v in r.items() if k.endswith("_eo")),default=1.0)
            dp_v=max((abs(v) for k,v in r.items() if k.endswith("_dp")),default=1.0)
            tag=("ZERO" if (floor2(eo_v)==0.0 and floor2(dp_v)==0.0) else
                 "EO=0" if floor2(eo_v)==0.0 else
                 "DP=0" if floor2(dp_v)==0.0 else "    ")
            print(f"  [{tag}] run {run}: acc={r.get('acc',0):.4f} "
                  f"EO_fl={floor2(eo_v):.2f} DP_fl={floor2(dp_v):.2f}")
        except Exception as e:
            print(f"  run {run} error: {e}")
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    return results, bacc


def run_all_datasets(datasets=None, n_runs=3, verbose=True,
                     continue_on_error=True, mode=PIPELINE_MODE):
    if datasets is None: datasets=list(_PREPROCESS.keys())
    print("="*65)
    print(f"  FAIRNESS PIPELINE v4  —  mode: {mode.upper()}")
    print(f"  EO train target: {EO_TRAIN_TARGET}  (early exit when hit)")
    print(f"  No GroupCalibrator  |  Anchored sweep when EO low")
    print(f"  Device: {DEVICE}  |  Datasets: {datasets}")
    print("="*65)
    cache={}; failed=[]
    for i,ds in enumerate(datasets,1):
        print(f"\n{'─'*65}\n  [{i}/{len(datasets)}] {ds.upper()}\n{'─'*65}")
        try:
            res,bacc=run_dataset(ds,n_runs=n_runs,verbose=verbose,mode=mode)
            cache[ds]=(res,bacc)
        except Exception as e:
            print(f"  FAILED {ds}: {e}"); failed.append(ds)
            if not continue_on_error: raise
    print(f"\n  Done: {[d for d in datasets if d not in failed]}")
    print(f"  Failed: {failed}")
    _print_final_table(cache, mode=mode)
    return cache


def _print_final_table(cache, mode=PIPELINE_MODE):
    import json
    print(f"\n{'='*70}")
    print(f"  FINAL RESULTS  (mode={mode}  |  v4)")
    print(f"  {'Dataset':12} {'Base':>7} {'Ours':>7} {'Drop':>7} "
          f"{'EO_fl':>6} {'DP_fl':>6}  Status")
    print("  "+"─"*58)
    out_data={}
    for ds,(rr,bacc) in cache.items():
        if not rr: print(f"  {ds.upper():12} no results"); continue
        avg={k:float(np.mean([r[k] for r in rr if k in r]))
             for k in rr[0] if isinstance(rr[0].get(k),(int,float,np.floating))}
        eo=max((abs(avg[k]) for k in avg if k.endswith('_eo')),default=1.0)
        dp=max((abs(avg[k]) for k in avg if k.endswith('_dp')),default=1.0)
        drop=bacc-avg.get('acc',0.0)
        ef,df=floor2(eo),floor2(dp)
        tag=("ZERO" if ef==0.0 and df==0.0 else
             "EO=0" if ef==0.0 else "DP=0" if df==0.0 else "FAIL")
        hi=" ← HIGH DROP" if drop>0.06 else ""
        print(f"  {ds.upper():12} {bacc:7.4f} {avg.get('acc',0):7.4f} "
              f"{drop:+7.4f} {ef:6.2f} {df:6.2f}  {tag}{hi}")
        out_data[ds]={"baseline_acc":bacc,"our_acc":avg.get('acc',0),
                      "eo":eo,"dp":dp,"n_runs":len(rr)}
    fn=f"final_results_{mode}_v4.json"
    with open(os.path.join(RESULTS_DIR,fn),"w") as f:
        json.dump({"mode":mode,"version":"v4","results":out_data},f,indent=2)
    print(f"\n  Saved → {os.path.join(RESULTS_DIR,fn)}")


print("Pipeline functions defined.")
print(f"\nCurrent mode: {PIPELINE_MODE}")
print(f"EO train target: {EO_TRAIN_TARGET}")
print("Key v4 differences from v3:")
print("  - GroupCalibrator removed (raised EO on Adult in v3)")
print("  - Checkpoint selection = min EO s.t. acc >= budget (cleaner)")
print("  - Anchored sweep: if EO < target, search tau in [0.44, 0.56] only")
print("  - Early exit from training when EO_TRAIN_TARGET hit")
print("  - eo_loss_weight boosted: Adult 4.0, Hospital 3.5, COMPAS 3.5")


results_cache = run_all_datasets(
    datasets=['adult', 'compas', 'german', 'bank', 'lawschool', 'hospital'],
    n_runs=3,
    verbose=True,
    continue_on_error=True,
    mode=PIPELINE_MODE,
)

print("\nDone.")

/usr/local/lib/python3.12/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


Device: cuda  |  CUDA: True
Pipeline mode: eo_only  |  EO train target: 0.008
Hyperparams defined. Mode: eo_only
Utility helpers defined.
All 6 preprocessing functions defined.
Neural network modules defined.
Threshold sweep defined.
train_model defined.
Pipeline functions defined.

Current mode: eo_only
EO train target: 0.008
Key v4 differences from v3:
  - GroupCalibrator removed (raised EO on Adult in v3)
  - Checkpoint selection = min EO s.t. acc >= budget (cleaner)
  - Anchored sweep: if EO < target, search tau in [0.44, 0.56] only
  - Early exit from training when EO_TRAIN_TARGET hit
  - eo_loss_weight boosted: Adult 4.0, Hospital 3.5, COMPAS 3.5
  FAIRNESS PIPELINE v4  —  mode: EO_ONLY
  EO train target: 0.008  (early exit when hit)
  No GroupCalibrator  |  Anchored sweep when EO low
  Device: cuda  |  Datasets: ['adult', 'compas', 'german', 'bank', 'lawschool', 'hospital']

─────────────────────────────────────────────────────────────────
  [1/6] ADULT
─────────────────────────

 eo only, dp only. should it be combined?

The key thing is to keep the architecture identical and only change the loss/scoring functions, so the comparison is controlled. (while running dp script)